# Implementação da Arquitetura LeNet-5 (Variante Moderna)

A LeNet-5 é uma arquitetura de rede neural convolucional clássica. Esta versão utiliza **Max Pooling** e a função de ativação **ReLU**.

### Estrutura do Modelo:
1. **Input**: Imagem 32x32 (ou 28x28 com padding).
2. **C1**: Camada Convolucional (6 filtros 5x5, stride 1).
3. **S2**: Max Pooling (2x2, stride 2).
4. **C3**: Camada Convolucional (16 filtros 5x5, stride 1).
5. **S4**: Max Pooling (2x2, stride 2).
6. **C5**: Camada Convolucional (120 filtros 5x5, stride 1).
7. **F6**: Camada Totalmente Conectada (84 neurônios).
8. **Output**: Camada Totalmente Conectada (10 neurônios).

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import matplotlib.pyplot as plt
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import ssl
import os
import numpy as np

# Fix para erros de download do MNIST (SSL)
ssl._create_default_https_context = ssl._create_unverified_context

# Garante que os gráficos apareçam no notebook
%matplotlib inline

class LeNet5(nn.Module):
    def __init__(self, num_classes=10):
        super(LeNet5, self).__init__()
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=6, kernel_size=5, stride=1)
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.conv2 = nn.Conv2d(in_channels=6, out_channels=16, kernel_size=5, stride=1)
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.conv3 = nn.Conv2d(in_channels=16, out_channels=120, kernel_size=5, stride=1)
        self.fc1 = nn.Linear(in_features=120, out_features=84)
        self.fc2 = nn.Linear(in_features=84, out_features=num_classes)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = self.pool1(x)
        x = F.relu(self.conv2(x))
        x = self.pool2(x)
        x = F.relu(self.conv3(x))
        x = torch.flatten(x, 1)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

# Configuração de dispositivo
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = LeNet5(num_classes=10).to(device)
print(f"Modelo instanciado no dispositivo: {device}")

### Carregamento do MNIST


In [ ]:
transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)
test_loader = DataLoader(test_dataset, batch_size=1000, shuffle=False)

print(f"Dataset pronto! {len(train_dataset)} imagens de treino e {len(test_dataset)} de teste.")

### Visualização dos Mapas de Características (Feature Maps)

Aqui visualizamos como a imagem fica **após** passar pela primeira camada convolucional (`C1`). É possível ver quais padrões cada filtro está extraindo da imagem original.

In [ ]:
def visualize_feature_maps(model, loader):
    model.eval()
    # Pega uma imagem do dataset
    data, target = next(iter(loader))
    data = data[0:1].to(device) # Pega apenas a primeira imagem e manda pro dispositivo
    
    # Passa a imagem APENAS pela primeira camada convolucional + ativação
    with torch.no_grad():
        feature_maps = F.relu(model.conv1(data))
    
    # Plotagem
    fig, axes = plt.subplots(1, 7, figsize=(15, 3))
    
    # Imagem Original
    img_orig = data.cpu().squeeze().numpy() * 0.3081 + 0.1307
    axes[0].imshow(img_orig, cmap='gray')
    axes[0].set_title("Original")
    axes[0].axis('off')
    
    # Feature Maps (6 canais na LeNet-5 C1)
    f_maps = feature_maps.cpu().squeeze().numpy()
    for i in range(6):
        axes[i+1].imshow(f_maps[i], cmap='viridis')
        axes[i+1].set_title(f"Mapa {i+1}")
        axes[i+1].axis('off')
        
    plt.suptitle("Saída da Primeira Camada Convolucional (C1)", fontsize=14)
    plt.tight_layout()
    plt.show()

print("Visualização com pesos iniciais (aleatórios):")
visualize_feature_maps(model, test_loader)

### Treinamento da Rede Neural


In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
epochs = 3

model.train()
last_loss = 0.0
for epoch in range(epochs):
    running_loss = 0.0
    for batch_idx, (data, target) in enumerate(train_loader):
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        outputs = model(data)
        loss = criterion(outputs, target)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    
    last_loss = running_loss / len(train_loader)
    print(f'--- Época {epoch+1} Finalizada | Loss: {last_loss:.4f} ---')

print("Treinamento Concluído!")

### Resultados PÓS-Treinamento

Vamos ver como os mapas de características mudaram após a rede aprender os padrões dos dígitos.

In [ ]:
print("Visualização com pesos treinados:")
visualize_feature_maps(model, test_loader)

### Analisando os Erros (Predições Incorretas)


In [ ]:
def plot_misclassified(model, loader, num_images=10):
    model.eval()
    misclassified_images = []
    actual_labels = []
    predicted_labels = []
    with torch.no_grad():
        for data, target in loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            pred = output.argmax(dim=1)
            mask = (pred != target)
            idxs = mask.nonzero(as_tuple=True)[0]
            for idx in idxs:
                misclassified_images.append(data[idx].cpu())
                actual_labels.append(target[idx].cpu().item())
                predicted_labels.append(pred[idx].cpu().item())
                if len(misclassified_images) >= num_images: break
            if len(misclassified_images) >= num_images: break

    plt.figure(figsize=(15, 6))
    for i in range(len(misclassified_images)):
        img = misclassified_images[i].squeeze().numpy() * 0.3081 + 0.1307
        plt.subplot(2, 5, i + 1)
        plt.imshow(img, cmap='gray')
        plt.title(f'Prev: {predicted_labels[i]}\nReal: {actual_labels[i]}', color='red')
        plt.axis('off')
    plt.tight_layout()
    plt.show()

plot_misclassified(model, test_loader)

### Salvando o Checkpoint

In [ ]:
os.makedirs('models', exist_ok=True)
checkpoint = {
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
}
torch.save(checkpoint, "models/lenet5_checkpoint.pth")
print("Checkpoint salvo em models/lenet5_checkpoint.pth")